# RepOpt-3D Colab Fusion Eval

This notebook bootstraps a Colab runtime for OpenScene fusion-mode evaluation without installing `MinkowskiEngine`.

Expected runtime:
- Colab with GPU enabled
- repo branch: `dev-khalit-reopt3d`
- OpenScene submodule branch: `dev-khalit-openscene`


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Fill these in before running the notebook.
REPO_URL = "https://github.com/a1bhinav/repopt-3d.git"
REPO_BRANCH = "dev-khalit-reopt3d"

OPENSCENE_FORK_URL = "https://github.com/St1p42/openscene.git"
OPENSCENE_BRANCH = "dev-khalit-openscene"

# Use Drive-mounted or other externally mounted storage.
DATA_ROOT = "/content/drive/MyDrive/repopt-data/matterport_3d"
FUSED_ROOT = "/content/drive/MyDrive/repopt-data/matterport_multiview_openseg_test"
SAVE_ROOT = "/content/drive/MyDrive/repopt-output/fusion_eval"

# Set to 1 for the first debug run.
TEST_REPEATS = 1


In [ ]:
!nvidia-smi

In [ ]:
%cd /content
!rm -rf repopt-3d
!git clone --recurse-submodules {REPO_URL}
%cd /content/repopt-3d
!git checkout {REPO_BRANCH}
!git submodule update --init --recursive
%cd /content/repopt-3d/third_party/openscene
!git remote set-url origin {OPENSCENE_FORK_URL}
!git fetch origin {OPENSCENE_BRANCH}
!git checkout {OPENSCENE_BRANCH}
%cd /content/repopt-3d


In [ ]:
!python -V
!pip install --upgrade pip
!pip install --no-cache-dir torch==2.4.1 torchvision==0.19.1 --index-url https://download.pytorch.org/whl/cu124
!pip install --no-cache-dir scipy open3d ftfy tensorboardx tqdm imageio plyfile opencv-python sharedarray git+https://github.com/openai/CLIP.git


In [ ]:
!python -c "import scipy, open3d, ftfy, tensorboardX, tqdm, imageio, plyfile, cv2, SharedArray, clip; print('All OpenScene deps OK')"

In [ ]:
import os
os.environ['DATA_ROOT'] = DATA_ROOT
os.environ['FUSED_ROOT'] = FUSED_ROOT
os.environ['SAVE_ROOT'] = SAVE_ROOT
os.environ['TEST_REPEATS'] = str(TEST_REPEATS)
os.makedirs(SAVE_ROOT, exist_ok=True)
print('DATA_ROOT =', DATA_ROOT)
print('FUSED_ROOT =', FUSED_ROOT)
print('SAVE_ROOT =', SAVE_ROOT)
print('DATA_ROOT exists =', os.path.exists(DATA_ROOT))
print('FUSED_ROOT exists =', os.path.exists(FUSED_ROOT))


For the first run, keep `TEST_REPEATS=1` and disable visualization. Once this works on your full dataset or subset, increase repeats as needed.

In [ ]:
%cd /content/repopt-3d/third_party/openscene
!python run/evaluate.py \
  --config=config/matterport/ours_openseg_pretrained.yaml \
  feature_type fusion \
  data_root {DATA_ROOT} \
  data_root_2d_fused_feature {FUSED_ROOT} \
  save_folder {SAVE_ROOT} \
  test_repeats {TEST_REPEATS} \
  vis_input False \
  vis_pred False \
  vis_gt False


In [ ]:
!ls -lah {SAVE_ROOT}
